# Introduction to SQL with Fantasy Premier League

**Learning Objective:** By the end of this notebook you will know how to use SQL to query a real database, and you will use those queries — plus Pandas — to find the best-value players and pick a Fantasy Premier League team within a £100 million budget.

**Economics Concept:** This notebook combines two skills that economists use every day. The first is **SQL (Structured Query Language)**, the standard tool for asking questions of large datasets stored in a database. The second is **value analysis**: comparing what you get (points) against what you pay (price), which economists call cost-effectiveness. By the end you will have built an automated team-picker that solves a classic **budget allocation problem** — getting the most total output from a fixed pool of resources.

In [ ]:
# Uncomment and run this cell ONCE if you have not installed these packages yet
# !pip install sqlalchemy jupysql

## Setting Up

Before we can query data, we need to import the libraries we will use. Think of this step as opening the right apps before you start working.

- **pandas** is the Python library for working with tables of data.
- **sqlalchemy** creates the connection between Python and a SQL database.
- **plotly** lets us draw interactive charts so we can mouse over data points and see player names.
- **`%load_ext sql`** switches on the SQL magic command, which lets us write SQL directly in notebook cells.

In [ ]:
import pandas as pd  # for working with tables of data in Python
import sqlalchemy  # for connecting Python to a SQL database
import plotly.graph_objects as go  # for creating interactive charts with hover tooltips
from plotly.subplots import make_subplots  # for arranging multiple charts in a grid layout
import datetime
import sqlite3
%load_ext sql


---
## Part 0 — Building the Database from Live Data

Before we can run any SQL queries, we need a database to query. A **database** is an organised collection of data stored in a structured format. In this notebook, we build our own database by downloading the latest Fantasy Premier League data directly from the internet.

We give the database a **date-stamped filename** — for example `fpl_03_04_26.db` — so that every time you run this notebook you create a fresh, clearly labelled snapshot of the current data. This mirrors a good practice in data work: always record *when* your data was collected.

In [ ]:
# Generate today's date to use in the database filename
# This means every run produces a file like fpl_03_04_26.db
database_creation_date = datetime.date.today()
db_filename = "fpl_{:02d}_{:02d}_{}.db".format(
    database_creation_date.month,
    database_creation_date.day,
    str(database_creation_date.year)[2:]
)

print("Today's date:", database_creation_date)
print("Database will be saved as:", db_filename)

### Step 1 — Download the Latest FPL Data

The FPL data lives on GitHub as four separate CSV files. We use **pandas** (`pd.read_csv`) to download each file directly from its URL into a Python table called a DataFrame.

- **`player_info_data`** — one row per player, with name, position, and team
- **`player_stats_data`** — detailed statistics per player updated each gameweek
- **`team_info_data`** — one row per Premier League club
- **`gameweek_summary_data`** — high-level totals for each gameweek

In [ ]:
# Base URL where the FPL CSV files are stored on GitHub
fpl_data_base_url = "https://raw.githubusercontent.com/olbauday/FPL-Core-Insights/main/data/2025-2026/"

# Download each CSV file into a pandas DataFrame
player_info_data = pd.read_csv(fpl_data_base_url + "players.csv")
player_stats_data = pd.read_csv(fpl_data_base_url + "playerstats.csv")
team_info_data = pd.read_csv(fpl_data_base_url + "teams.csv")
gameweek_summary_data = pd.read_csv(fpl_data_base_url + "gameweek_summaries.csv")

print("Downloaded", len(player_info_data), "players")
print("Downloaded", len(team_info_data), "teams")
print("Downloaded", len(player_stats_data), "rows of player statistics")
print("Downloaded", len(gameweek_summary_data), "gameweek summary rows")

### Step 2 — Save the Data to a SQLite Database

We now write the four DataFrames into a **SQLite database** — a single file on your computer that stores the data in a structured, queryable format.

`sqlite3.connect(db_filename)` creates (or opens) the database file. The `.to_sql()` method writes each DataFrame as a named table inside that file. Setting `if_exists='replace'` means that if you run this cell again the old data is overwritten with fresh data, so your database is always up to date.

In [ ]:
# Create the SQLite database file and write all four tables into it
fpl_db_conn = sqlite3.connect(db_filename)

player_info_data.to_sql('players', fpl_db_conn, if_exists='replace', index=False)
player_stats_data.to_sql('playerstats', fpl_db_conn, if_exists='replace', index=False)
team_info_data.to_sql('teams', fpl_db_conn, if_exists='replace', index=False)
gameweek_summary_data.to_sql('gameweek_summaries', fpl_db_conn, if_exists='replace', index=False)

fpl_db_conn.close()
print("Database", db_filename, "is ready with fresh data.")

## Connecting to the Fantasy Premier League Database

The database we just built is stored in the date-stamped file whose name was printed above. SQLite is a type of database stored as a single file on your computer — perfect for learning.

The line below creates an **engine**, which is the connection between Python and the database. Once it is made, every `%sql` or `%%sql` cell in this notebook will send its query directly to that database.

In [ ]:
# Create a connection to the date-stamped Fantasy Premier League database file
fpl_engine = sqlalchemy.create_engine('sqlite:///' + db_filename)
fpl_connection = fpl_engine.connect()

In [ ]:
# Tell the SQL magic extension which database to use
%sql fpl_engine

---
## Part 1 — Exploring the Database

### What Tables Are in the Database?

A SQL database is organised into **tables**. Each table is like one sheet in a spreadsheet. The special table `sqlite_master` is a built-in SQLite table that stores information about the database itself.

The query below uses `SELECT` to pick the `name` column and `WHERE` to filter for rows that describe tables (not indexes or other objects).

In [ ]:
%%sql
SELECT name
FROM sqlite_master
WHERE type = 'table';

There are four tables:

| Table | What it contains |
|---|---|
| `players` | One row per player — name, team, position |
| `playerstats` | Detailed statistics for every player, updated each gameweek |
| `teams` | One row per Premier League team |
| `gameweek_summaries` | Summary information for each gameweek of the season |

### Inspecting the `players` Table

`PRAGMA table_info()` is a SQLite command that lists every column in a table — like reading the header row of a spreadsheet. It tells you each column's name and data type.

In [ ]:
%sql PRAGMA table_info(players);

Now let's see the actual rows. `SELECT *` means "give me every column". `LIMIT 5` means "stop after the first 5 rows" — useful when a table has thousands of rows and you just want a quick peek.

In [ ]:
%sql SELECT * FROM players LIMIT 5;

### Inspecting the `teams` Table

Next let's look at the teams. This table has one row for every Premier League club. Let's fetch all 20 teams and just their name and short name columns — we don't need every column right now.

In [ ]:
%sql SELECT name, short_name, strength FROM teams ORDER BY strength DESC;

The `strength` column is a FPL difficulty rating — higher numbers mean stronger teams. You can see the traditional "big clubs" cluster at the top.

### How Many Players Are in Each Position?

FPL has four positions: Goalkeeper, Defender, Midfielder, and Forward. Let's count how many players exist in each. We use `COUNT(*)` to count rows, and `GROUP BY` to count separately for each position.

`GROUP BY` is one of the most powerful SQL commands. It works exactly like a pivot table: it groups all rows that share the same value in a column, then applies an aggregation function (like `COUNT` or `AVG`) to each group.

In [ ]:
%%sql
SELECT position, COUNT(*) AS number_of_players
FROM players
GROUP BY position
ORDER BY number_of_players DESC;

---
## Part 2 — Filtering and Sorting Players

### WHERE: Filter to One Position

The `WHERE` clause filters rows — only rows that satisfy the condition are returned. Here we ask for all Forwards in the database.

In [ ]:
%%sql
SELECT first_name, second_name, web_name, position
FROM players
WHERE position = 'Forward'
LIMIT 10;

### Exploring `playerstats`: Price and Points

The `playerstats` table is updated each gameweek. The key columns we care about are:

- **`now_cost`** — current price in millions of pounds (e.g. 14.7 means £14.7 million)
- **`total_points`** — total FPL points scored so far this season
- **`points_per_game`** — average FPL points per match played
- **`gw`** — the gameweek this row was recorded in

Let's check what the latest gameweek is.

In [ ]:
%sql SELECT MAX(gw) AS latest_gameweek FROM playerstats;

Now let's look at the most expensive players in the latest gameweek. We filter using `WHERE gw = 29` so we only see the most up-to-date snapshot, then sort by price descending.

In [ ]:
%%sql
SELECT web_name, now_cost, total_points, points_per_game
FROM playerstats
WHERE gw = 29
ORDER BY now_cost DESC
LIMIT 10;

Expensive players are not always the best performers. Let's now look at who has scored the **most total points** this season.

In [ ]:
%%sql
SELECT web_name, now_cost, total_points, points_per_game
FROM playerstats
WHERE gw = 29
ORDER BY total_points DESC
LIMIT 10;

---
## Part 3 — Aggregation: Averages Across Positions

### Average Price and Points by Position

We now join `playerstats` with `players` so we can group by position. A **JOIN** combines two tables using a shared column — here `playerstats.id` matches `players.player_id`.

Think of a JOIN like a VLOOKUP in Excel: it looks up matching rows in another table and pulls in additional columns.

We also use `ROUND()` to limit decimal places — keeping the output easy to read.

In [ ]:
%%sql
SELECT
    p.position,
    COUNT(*) AS num_players,
    ROUND(AVG(ps.now_cost), 1) AS avg_price_millions,
    ROUND(AVG(ps.total_points), 0) AS avg_total_points
FROM playerstats ps
JOIN players p ON ps.id = p.player_id
WHERE ps.gw = 29
GROUP BY p.position
ORDER BY avg_total_points DESC;

This tells us that Defenders and Midfielders have similar average points but Defenders cost noticeably less on average. That gap hints at where the best value might be — exactly the kind of insight economists call a **market inefficiency**.

### Top Points Scorers Per Team

Let's find out which team produces the most FPL points on average. We join all three tables: `playerstats`, `players`, and `teams`.

In [ ]:
%%sql
SELECT
    t.name AS team_name,
    ROUND(AVG(ps.total_points), 1) AS avg_player_points,
    ROUND(AVG(ps.now_cost), 1) AS avg_player_price
FROM playerstats ps
JOIN players p ON ps.id = p.player_id
JOIN teams t ON p.team_code = t.code
WHERE ps.gw = 29
GROUP BY t.name
ORDER BY avg_player_points DESC
LIMIT 10;

---
## Part 4 — Value Analysis: Points Per Million

### What Is Value?

In FPL, the best managers do not just buy the highest-scoring players — they find players who deliver a lot of points *relative to their price*. This is the **value** of a player.

Economists call this concept **cost-effectiveness**: how much output do you get for each unit of input? Here our output is FPL points and our input is price.

We calculate it as:

> **Points per million = total_points ÷ now_cost**

The database already has a `value_season` column that tracks this. But we can also compute it ourselves with SQL arithmetic. Let's do both and see that they match — a good way to check our understanding.

In [ ]:
%%sql
SELECT
    p.web_name,
    p.position,
    ps.now_cost,
    ps.total_points,
    ps.value_season AS value_from_db,
    ROUND(ps.total_points * 1.0 / ps.now_cost, 1) AS value_we_calculated
FROM playerstats ps
JOIN players p ON ps.id = p.player_id
WHERE ps.gw = 29
ORDER BY ps.total_points DESC
LIMIT 10;

The two value columns agree — great. Now let's find the **best-value players by position**. We filter for players with at least 200 minutes played (`minutes > 200`) so we are not misled by players with very few appearances.

In [ ]:
%%sql
SELECT
    p.web_name,
    p.position,
    ps.now_cost,
    ps.total_points,
    ROUND(ps.total_points * 1.0 / ps.now_cost, 2) AS points_per_million
FROM playerstats ps
JOIN players p ON ps.id = p.player_id
WHERE ps.gw = 29
  AND ps.minutes > 200
ORDER BY points_per_million DESC
LIMIT 15;

Notice how most of the top-value players are Defenders — cheap players who quietly rack up points through clean sheets and bonus. Haaland, the most expensive player, does not appear here because his high price dilutes his value score.

### Best Value Per Position

Let's look at the top-value player in each position separately, by adding a `WHERE position = '...'` filter. We start with Goalkeepers — the cheapest slot in the squad.

In [ ]:
%%sql
SELECT
    p.web_name,
    p.position,
    ps.now_cost,
    ps.total_points,
    ROUND(ps.total_points * 1.0 / ps.now_cost, 2) AS points_per_million
FROM playerstats ps
JOIN players p ON ps.id = p.player_id
WHERE ps.gw = 29
  AND ps.minutes > 200
  AND p.position = 'Goalkeeper'
ORDER BY points_per_million DESC
LIMIT 10;

Now we repeat the same query for Defenders — the key building block of a value-oriented team.

In [ ]:
%%sql
SELECT
    p.web_name,
    p.position,
    ps.now_cost,
    ps.total_points,
    ROUND(ps.total_points * 1.0 / ps.now_cost, 2) AS points_per_million
FROM playerstats ps
JOIN players p ON ps.id = p.player_id
WHERE ps.gw = 29
  AND ps.minutes > 200
  AND p.position = 'Defender'
ORDER BY points_per_million DESC
LIMIT 10;

Now Midfielders — the position with the most players and often where the biggest scores come from.

In [ ]:
%%sql
SELECT
    p.web_name,
    p.position,
    ps.now_cost,
    ps.total_points,
    ROUND(ps.total_points * 1.0 / ps.now_cost, 2) AS points_per_million
FROM playerstats ps
JOIN players p ON ps.id = p.player_id
WHERE ps.gw = 29
  AND ps.minutes > 200
  AND p.position = 'Midfielder'
ORDER BY points_per_million DESC
LIMIT 10;

Finally, Forwards — typically the most expensive position but essential for attacking output.

In [ ]:
%%sql
SELECT
    p.web_name,
    p.position,
    ps.now_cost,
    ps.total_points,
    ROUND(ps.total_points * 1.0 / ps.now_cost, 2) AS points_per_million
FROM playerstats ps
JOIN players p ON ps.id = p.player_id
WHERE ps.gw = 29
  AND ps.minutes > 200
  AND p.position = 'Forward'
ORDER BY points_per_million DESC
LIMIT 10;

---
## Part 5 — From SQL to Pandas

### Pulling SQL Results Into Python

So far we have been reading results inside the notebook. Now we want to **work with the data in Python** so we can do more complex calculations — like building a full team.

The `%%sql result_name <<` syntax saves a SQL result to a Python variable. We then call `.DataFrame()` to convert it into a Pandas DataFrame, which is the standard Python table format.

We pull all players from the latest gameweek snapshot who have played more than 200 minutes.

In [ ]:
%%sql fpl_player_data <<
SELECT
    p.player_id,
    p.web_name,
    p.position,
    t.name AS team_name,
    t.short_name AS team_short,
    ps.now_cost,
    ps.total_points,
    ps.points_per_game,
    ps.minutes,
    ps.goals_scored,
    ps.assists,
    ps.clean_sheets,
    ps.value_season
FROM playerstats ps
JOIN players p ON ps.id = p.player_id
JOIN teams t ON p.team_code = t.code
WHERE ps.gw = 29
  AND ps.minutes > 200
ORDER BY p.position, ps.total_points DESC;

In [ ]:
# Convert the SQL result to a Pandas DataFrame
all_players_data = fpl_player_data.DataFrame()

# Add a points_per_million column so we can rank by value
all_players_data['points_per_million'] = all_players_data['total_points'] / all_players_data['now_cost']
all_players_data['points_per_million'] = all_players_data['points_per_million'].round(2)

print(f"Total qualified players: {len(all_players_data)}")
all_players_data.head(10)

### Visualising Value by Position

A scatter plot of price vs points lets us see the value landscape visually. Players above the trend line are **better value** than average — they score more points per pound spent. Players below the line are **overpriced** relative to their output.

This kind of chart is called a **production frontier** in economics: it shows the maximum output achievable at each cost level.

In [ ]:
# Set up a 2x2 grid of interactive charts, one per position
position_list = ['Goalkeeper', 'Defender', 'Midfielder', 'Forward']
row_list = [1, 1, 2, 2]
col_list = [1, 2, 1, 2]
color_list = ['green', 'blue', 'orange', 'red']

position_chart = make_subplots(
    rows=2, cols=2,
    subplot_titles=position_list
)

for i in range(len(position_list)):
    current_position = position_list[i]
    current_row = row_list[i]
    current_col = col_list[i]
    current_color = color_list[i]

    position_players_data = all_players_data[all_players_data['position'] == current_position]

    position_chart.add_trace(
        go.Scatter(
            x=position_players_data['now_cost'],
            y=position_players_data['total_points'],
            mode='markers',
            marker=dict(color=current_color, opacity=0.6, size=8),
            text=position_players_data['web_name'],
            hovertemplate='<b>%{text}</b><br>Price: £%{x}m<br>Points: %{y}<extra></extra>',
            name=current_position,
            showlegend=False
        ),
        row=current_row,
        col=current_col
    )

position_chart.update_xaxes(title_text='Price (£m)')
position_chart.update_yaxes(title_text='Total Points')
position_chart.update_layout(
    title_text='Price vs Total Points by Position (players with >200 minutes)',
    height=700
)
position_chart.show()

The labelled names are the best-value players in each position — the ones that sit highest on the chart relative to their price. Notice that some cheap Defenders score nearly as many points as Midfielders who cost twice as much.

---
## Part 6 — The Team Builder: Picking Your Best XI

### The Rules

Fantasy Premier League gives you a **£100 million budget** to build a squad of 15 players. A standard starting lineup follows this formation:

| Position | Starters | Total in squad |
|---|---|---|
| Goalkeeper | 1 | 2 |
| Defender | 4 | 5 |
| Midfielder | 4 | 5 |
| Forward | 2 | 3 |

This is a classic **constrained optimisation problem**: maximise total points subject to a budget constraint and fixed position quotas. Economists solve problems like this every day — allocating a government budget across departments, a firm's R&D across projects, or a portfolio across assets.

Our strategy: for each position we take the players with the **highest points-per-million value**, then check we stay within budget. This greedy approach is a good starting point.

In [ ]:
# Define the squad structure for a 4-4-2 formation
total_budget_millions = 100.0

starters_needed = {
    'Goalkeeper': 1,
    'Defender': 4,
    'Midfielder': 4,
    'Forward': 2
}

squad_needed = {
    'Goalkeeper': 2,
    'Defender': 5,
    'Midfielder': 5,
    'Forward': 3
}

print("Budget: £{:.1f}m".format(total_budget_millions))
print("Formation: 4-4-2")
print("Squad size: 15")

In [ ]:
# Sort all players by points_per_million so the best-value players come first
all_players_ranked = all_players_data.sort_values('points_per_million', ascending=False)
all_players_ranked = all_players_ranked.reset_index(drop=True)

print("Top 5 best-value players overall:")
print(all_players_ranked[['web_name', 'position', 'now_cost', 'total_points', 'points_per_million']].head())

### Picking the Starting XI

We loop through each position and pick the top-value players one at a time, keeping track of the total cost. This is called a **greedy algorithm**: at each step, take the locally best option available.

In [ ]:
# Pick the starting XI based on best value per position
starting_xi_rows = []
total_starting_cost = 0.0

for current_position in ['Goalkeeper', 'Defender', 'Midfielder', 'Forward']:
    # Get all players at this position, sorted by value
    position_players_sorted = all_players_ranked[all_players_ranked['position'] == current_position]

    # Take the top N starters for this position
    number_needed = starters_needed[current_position]
    chosen_starters = position_players_sorted.head(number_needed)

    for row_index, player_row in chosen_starters.iterrows():
        starting_xi_rows.append(player_row)
        total_starting_cost = total_starting_cost + player_row['now_cost']

starting_xi_data = pd.DataFrame(starting_xi_rows)
starting_xi_data = starting_xi_data.reset_index(drop=True)

print("=== Starting XI (best value by position) ===")
print(starting_xi_data[['web_name', 'position', 'team_short', 'now_cost', 'total_points', 'points_per_million']].to_string(index=False))
print()
print("Total starting XI cost: £{:.1f}m".format(total_starting_cost))
print("Total starting XI points: {}".format(int(starting_xi_data['total_points'].sum())))

### Completing the Squad with Bench Players

We now fill the 4 bench spots (1 extra GK, 1 extra DEF, 1 extra MID, 1 extra FWD). We pick the cheapest available players per position who were **not already selected as starters**, so we keep as much budget as possible for the starting XI.

In [ ]:
# Get the player IDs already picked in the starting XI
starting_xi_ids = set(starting_xi_data['player_id'].tolist())

# Pick the cheapest bench players (not already in the starting XI)
bench_players_rows = []
total_bench_cost = 0.0

for current_position in ['Goalkeeper', 'Defender', 'Midfielder', 'Forward']:
    bench_spots_for_position = squad_needed[current_position] - starters_needed[current_position]

    # Players at this position NOT already in the starting XI
    available_for_bench = all_players_ranked[
        (all_players_ranked['position'] == current_position) &
        (~all_players_ranked['player_id'].isin(starting_xi_ids))
    ]

    # Pick the cheapest available players for the bench
    bench_picks = available_for_bench.nsmallest(bench_spots_for_position, 'now_cost')

    for row_index, player_row in bench_picks.iterrows():
        bench_players_rows.append(player_row)
        total_bench_cost = total_bench_cost + player_row['now_cost']

bench_data = pd.DataFrame(bench_players_rows)
bench_data = bench_data.reset_index(drop=True)

print("=== Bench Players ===")
print(bench_data[['web_name', 'position', 'team_short', 'now_cost', 'total_points']].to_string(index=False))
print()
print("Bench cost: £{:.1f}m".format(total_bench_cost))

### Budget Check

Now we check whether the full 15-player squad stays within the £100 million budget.

In [ ]:
total_squad_cost = total_starting_cost + total_bench_cost
remaining_budget = total_budget_millions - total_squad_cost

print("=== Budget Summary ===")
print("Starting XI cost:   £{:.1f}m".format(total_starting_cost))
print("Bench cost:         £{:.1f}m".format(total_bench_cost))
print("Total squad cost:   £{:.1f}m".format(total_squad_cost))
print("Budget remaining:   £{:.1f}m".format(remaining_budget))
print()

if total_squad_cost <= total_budget_millions:
    print("✅ Squad is within the £100m budget!")
else:
    print("❌ Over budget by £{:.1f}m — you need to swap in cheaper players.".format(total_squad_cost - total_budget_millions))

### Full Squad Summary

Let's print the complete squad in a readable format, grouped by position.

In [ ]:
# Add a 'role' column to distinguish starters from bench
starting_xi_data['role'] = 'Starter'
bench_data['role'] = 'Bench'

# Combine into one full squad DataFrame
full_squad_data = pd.concat([starting_xi_data, bench_data], ignore_index=True)

print("=== FULL SQUAD — Best Value FPL Team ===")
print()

for current_position in ['Goalkeeper', 'Defender', 'Midfielder', 'Forward']:
    position_in_squad = full_squad_data[full_squad_data['position'] == current_position]
    print(f"--- {current_position}s ---")
    for row_index, player_row in position_in_squad.iterrows():
        role_label = "[START]" if player_row['role'] == 'Starter' else "[BENCH]"
        print("  {} {:20s} {:4s}  £{:.1f}m  {} pts  ({:.1f} pts/£m)".format(
            role_label,
            player_row['web_name'],
            player_row['team_short'],
            player_row['now_cost'],
            int(player_row['total_points']),
            player_row['points_per_million']
        ))
    print()

print("Total squad cost:  £{:.1f}m / £{:.1f}m budget".format(total_squad_cost, total_budget_millions))
print("Starting XI points: {}".format(int(starting_xi_data['total_points'].sum())))

### Points vs Price: Your Team on the Chart

Let's put your selected starters on the scatter chart from before so you can see how they compare to all other players.

In [ ]:
# Create an interactive scatter chart comparing all players to your starting XI
# Mouse over any dot to see the player name, price, and total points
team_comparison_chart = go.Figure()

# Plot all qualified players in grey
team_comparison_chart.add_trace(go.Scatter(
    x=all_players_data['now_cost'],
    y=all_players_data['total_points'],
    mode='markers',
    marker=dict(color='grey', opacity=0.3, size=6),
    text=all_players_data['web_name'],
    hovertemplate='<b>%{text}</b><br>Price: £%{x}m<br>Points: %{y}<extra></extra>',
    name='All players'
))

# Highlight the starting XI in gold
team_comparison_chart.add_trace(go.Scatter(
    x=starting_xi_data['now_cost'],
    y=starting_xi_data['total_points'],
    mode='markers',
    marker=dict(color='gold', size=12, line=dict(color='black', width=1)),
    text=starting_xi_data['web_name'],
    hovertemplate='<b>%{text}</b><br>Price: £%{x}m<br>Points: %{y}<extra></extra>',
    name='Your starting XI'
))

team_comparison_chart.update_layout(
    title='Your Best-Value Starting XI vs All Players',
    xaxis_title='Price (£m)',
    yaxis_title='Total FPL Points',
    height=600
)
team_comparison_chart.show()

Your selected players are shown in gold. Notice that most of them cluster in the lower-left part of the chart — cheap players who punch above their weight in points. That is the essence of value investing in any market, whether you are picking footballers or buying stocks.

---
## Summary

In this notebook you learned how to:

1. **Connect Python to a SQL database** using `sqlalchemy` and `%load_ext sql`.
2. **Explore a database** with `SELECT`, `LIMIT`, and `PRAGMA table_info()`.
3. **Filter rows** with `WHERE` and **sort results** with `ORDER BY`.
4. **Aggregate data** with `COUNT`, `AVG`, and `GROUP BY` — like a pivot table in SQL.
5. **Join tables** to combine player names, positions, and team names in one query.
6. **Calculate value** as points per million — a simple cost-effectiveness metric.
7. **Transfer SQL results into Pandas** using the `<<` syntax, then apply Python logic.
8. **Solve a budget allocation problem** — pick the best team within a £100m constraint.

The economic lesson: **prices in markets are not always perfectly aligned with underlying value**. Some players deliver far more output per pound than others, and systematic data analysis can reveal these gaps — the same principle applies to labour markets, financial assets, and government programmes. SQL and Pandas are two of the most important tools economists use to find those gaps.